# Connectivity-only InChIKey catalogue planning: Boceprevir

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Laboratoire-de-Chemoinformatique/SynPlanner/blob/main/tutorials/20_InChIKey_Building_Block_Catalogue.ipynb)

This tutorial uses an **already prepared**, vendor-aware JSON building-block catalogue. It shows the single immutable runtime catalogue used by MCTS, connectivity-only search for Boceprevir, and vendor cost calculation on a detached `Route`. It does not prepare or modify the catalogue. Full InChIKeys and `has_stereo` remain available as catalogue metadata, but current planning uses only the first 14 InChIKey characters.


## 1. Load the processed catalogue

Set SYNPLAN_BUILDING_BLOCKS_JSON to the prepared JSON file, or place it at the default path below. In Colab, upload or mount the file first. The GPS preset supplies the trained ranking policy and reaction rules, but its legacy SMILES stock is deliberately not used here.

In [ ]:
import os
from pathlib import Path

from synplan.utils.loading import download_selected_files

catalogue_path = Path(
    os.environ.get(
        "SYNPLAN_BUILDING_BLOCKS_JSON",
        "synplan_data/building_blocks/all-bb-2026-06/building_blocks.json",
    )
)
if not catalogue_path.is_file():
    raise FileNotFoundError(
        f"Prepared catalogue not found at {catalogue_path}. "
        "Set SYNPLAN_BUILDING_BLOCKS_JSON or run "
        "`synplan building_blocks_standardizing --input <catalogue.tsv> "
        "--output <catalogue.json>` first."
    )

# These are the reaction_rules and ranking_policy entries declared by the
# current synplanner-gps preset. Its legacy building-block stock is deliberately
# not downloaded or used.
data_root = download_selected_files(
    [
        ("policy/supervised_gps/v1", "reaction_rules.tsv"),
        ("policy/supervised_gps/v1/v1", "ranking_policy.ckpt"),
    ],
    save_to="synplan_data",
    extract_zips=False,
)
reaction_rules_path = data_root / "policy/supervised_gps/v1/reaction_rules.tsv"
ranking_policy_path = (
    data_root / "policy/supervised_gps/v1/v1/ranking_policy.ckpt"
)


In [ ]:
from synplan.chem.building_blocks import (
    load_building_block_catalogue,
    match_building_blocks,
)

building_blocks = load_building_block_catalogue(catalogue_path)

{
    "full_inchikey_records": sum(map(len, building_blocks.values())),
    "connectivity_buckets": len(building_blocks),
}

In [ ]:
example_prefix, example_bucket = next(iter(building_blocks.items()))
example_block = example_bucket[0]

{
    "prefix": example_prefix,
    "inchikey": example_block.inchikey,
    "smiles": example_block.smiles,
    "vendors": dict(example_block.vendors),
    "has_stereo": example_block.has_stereo,
    "candidate_count_for_first_14_characters": len(example_bucket),
}

## 2. Prepare Boceprevir for connectivity-only search

The input SMILES contains explicit tetrahedral stereochemistry, but current SynPlanner planning is intentionally stereo-agnostic. The normal `clean_stereo=True` path removes those descriptors before MCTS. Chython therefore produces `LHHCSNFAOIFYRV-UHFFFAOYSA-N`, and catalogue membership uses only its connectivity block, `LHHCSNFAOIFYRV`.


In [ ]:
import warnings

from IPython.display import display

from synplan.chem.building_blocks import molecule_has_stereo, molecule_to_inchikey
from synplan.chem.utils import StereoDiscardedWarning, mol_from_smiles

BOCEPREVIR_SMILES = (
    "CC1([C@@H]2[C@H]1[C@H](N(C2)C(=O)[C@H](C(C)(C)C)"
    "NC(=O)NC(C)(C)C)C(=O)NC(CC3CCC3)C(=O)C(=O)N)C"
)
BOCEPREVIR_INCHIKEY = "LHHCSNFAOIFYRV-UHFFFAOYSA-N"
BOCEPREVIR_CONNECTIVITY = "LHHCSNFAOIFYRV"

with warnings.catch_warnings():
    warnings.simplefilter("ignore", StereoDiscardedWarning)
    target_molecule = mol_from_smiles(BOCEPREVIR_SMILES, clean_stereo=True)
target_inchikey = molecule_to_inchikey(target_molecule)

assert not molecule_has_stereo(target_molecule)
assert target_inchikey == BOCEPREVIR_INCHIKEY
assert target_inchikey[:14] == BOCEPREVIR_CONNECTIVITY
display(target_molecule)
{
    "name": "Boceprevir",
    "canonical_smiles": str(target_molecule),
    "connectivity_inchikey": target_inchikey,
    "connectivity_block": target_inchikey[:14],
    "has_stereo_after_planning_cleanup": molecule_has_stereo(target_molecule),
    "catalogue_candidate_count": len(
        match_building_blocks(building_blocks, target_inchikey)
    ),
}


## 3. Run connectivity-only MCTS

The tree and rollout evaluator receive the same immutable prefix-bucket catalogue object. Each `BuildingBlock` in a bucket retains its full InChIKey, stereo flag, and vendor offers, but stock membership always accepts the complete 14-character connectivity bucket. The ranking model and rules are the normal GPS planning resources; only stock identity changes.


In [ ]:
from synplan.utils.loading import load_policy_function, load_reaction_rules

reaction_rules = load_reaction_rules(reaction_rules_path)
policy_function = load_policy_function(weights_path=ranking_policy_path)

In [ ]:
from synplan.mcts.config import RolloutEvaluationConfig, TreeConfig
from synplan.utils.loading import load_evaluation_function

tree_config = TreeConfig(
    search_strategy="expansion_first",
    max_iterations=300,
    max_time=120,
    max_depth=9,
    min_mol_size=1,
    init_node_value=0.5,
    ucb_type="uct",
    c_ucb=0.1,
)

evaluation_config = RolloutEvaluationConfig(
    policy_network=policy_function,
    reaction_rules=reaction_rules,
    building_blocks=building_blocks,
    min_mol_size=tree_config.min_mol_size,
    max_depth=tree_config.max_depth,
    normalize=True,
)
evaluation_function = load_evaluation_function(evaluation_config)

In [ ]:
from synplan.mcts.tree import Tree

tree = Tree(
    target=target_molecule,
    config=tree_config,
    reaction_rules=reaction_rules,
    building_blocks=building_blocks,
    expansion_function=policy_function,
    evaluation_function=evaluation_function,
)

assert tree.building_blocks is building_blocks
assert evaluation_function.rollout.building_blocks is building_blocks
assert not hasattr(tree, "use_full_inchikey")
assert not hasattr(evaluation_function.rollout, "use_full_inchikey")
tree.run()


## 4. Select and cost a Route

Solved routes are preferred. A bounded search can finish before solving Boceprevir; in that case the best unfinished route is still a valid detached Route, and costing it explicitly reports missing or unpriced leaves instead of inventing a complete total.

In [ ]:
from synplan.chem.reaction.routes import Route

solved_routes = tree.routes()
if solved_routes:
    route_status = "solved"
    selected_route = solved_routes[0]
else:
    unfinished_routes = tree.routes(solved_only=False)
    if not unfinished_routes:
        raise RuntimeError("The bounded search did not produce an expandable route.")
    route_status = "unfinished"
    selected_route = unfinished_routes[0]

assert isinstance(selected_route, Route)

{
    "route_status": route_status,
    "tree_node_id": selected_route.provenance.tree_node_id,
    "steps": len(selected_route),
    "leaves": len(selected_route.leaves()),
}

In [ ]:
leaf_matches = []
for leaf in selected_route.leaves():
    leaf_key = molecule_to_inchikey(leaf)
    candidates = match_building_blocks(building_blocks, leaf_key)
    leaf_matches.append(
        {
            "smiles": str(leaf),
            "leaf_inchikey": leaf_key,
            "connectivity_block": leaf_key[:14],
            "catalogue_candidate_count": len(candidates),
            "candidate_full_inchikeys": [
                candidate.inchikey for candidate in candidates
            ],
        }
    )

leaf_matches


In [ ]:
import json

cost = selected_route.calculate_cost(building_blocks)
json.dumps(cost)  # Route costs are ready for a JSON sidecar.
cost_label = (
    "complete solved-route cost"
    if route_status == "solved"
    else "incomplete cost for the best unfinished route"
)

{
    "route_status": route_status,
    "cost_label": cost_label,
    "complete_cost": cost["complete"],
    "cost_per_mol": cost["cost_per_mol"],
    "cost_per_gram": cost["cost_per_gram"],
    "priced_cost_per_mol": cost["priced_cost_per_mol"],
    "priced_cost_per_gram": cost["priced_cost_per_gram"],
    "cost_units": cost["cost_units"],
    "missing_leaves": cost["missing_leaves"],
    "unpriced_leaves": cost["unpriced_leaves"],
}

In [ ]:
import pandas as pd

pd.DataFrame(cost["leaves"])

## Takeaways

- The JSON is keyed by full Chython Standard InChIKeys and stores canonical SMILES, stereo presence, and positive vendor prices. Distinct stereoisomers remain distinct catalogue records.
- Current planning removes target and precursor stereo. Boceprevir therefore uses `LHHCSNFAOIFYRV-UHFFFAOYSA-N`, and every MCTS stock check uses the `LHHCSNFAOIFYRV` connectivity block.
- `has_stereo` and each record's complete InChIKey are retained only as future-facing catalogue metadata; they do not select a search policy.
- `Route.calculate_cost(building_blocks)` uses the same connectivity bucket as MCTS and chooses its cheapest positive offer, regardless of target stereo. It does not mutate the route or retain the catalogue.
- Missing and unpriced leaves keep incomplete results explicit. Costing assumes one molar equivalent per leaf occurrence and 100% reaction yield; prices remain unnormalized raw price-per-gram values.
